In [5]:
from pynq import allocate
import numpy as np
from pynq import Overlay

N = 32  # or your FFT size
in_buffer = allocate(shape=(N,), dtype=np.complex64)
out_buffer = allocate(shape=(N,), dtype=np.complex64)


In [6]:
ol = Overlay("fin.bit")
fft_ip = ol.FFT_0




In [7]:
length =32
inpt = allocate(shape=(length ,), dtype=np.int32)
outpt = allocate(shape=(length ,), dtype=np.int32)

In [8]:
f = open("dataINcpp.txt", 'r')

In [9]:
inp_vals=[x.split() for x in f.readlines()]
new=[]

In [10]:
for i in range(length):
    xr=int(float(inp_vals[i][0])*256)%(1<<16)
    xi=int(float(inp_vals[i][1])*256)%(1<<16)
    x=xi * (1<<16) + xr 
    new.append(x)
print(new)


[6685123, 37552378, 4278583774, 4292477171, 6946790, 24379428, 2031810, 5570673, 4291559806, 4280680528, 11009395, 4282581213, 4270588485, 4291887115, 24641928, 6291495, 4261805853, 2686887, 20119866, 4289986461, 4271242996, 32767564, 4287692670, 13106880, 4291493475, 6553371, 4275240830, 7208953, 5046289, 4289003358, 4289003348, 4266131248]


In [11]:
np.copyto(inpt, new)

In [12]:
fft_ip.write(0x10, inpt.physical_address)
fft_ip.write(0x18, outpt.physical_address)
fft_ip.write(0x0, 0x1)



In [13]:
cnt=0
while (fft_ip.read(0)>>1)&1!=0:
    cnt+=1
cnt

1

In [14]:
ret=np.zeros(32)
np.copyto(ret, outpt)
outpt , inpt , 

(PynqBuffer([   2031849, -187497689,   -9171840,  -85065392,  -44237580,
              -68220337, -129040663,  206702394,  -43776362,   47121192,
              -11403153,   32312471,  -70909925,   21562841,   46664239,
              -12649513, -125106877,   20513851,    6356580,   -1377272,
               85917692,  -53280373, -211093449,  265814198, -112002302,
               91880634,  169802045,  101123177,   40370489,   30538825,
              -18611687,  229376769]),
 PynqBuffer([  6685123,  37552378, -16383522,  -2490125,   6946790,
              24379428,   2031810,   5570673,  -3407490, -14286768,
              11009395, -12386083, -24378811,  -3080181,  24641928,
               6291495, -33161443,   2686887,  20119866,  -4980835,
             -23724300,  32767564,  -7274626,  13106880,  -3473821,
               6553371, -19726466,   7208953,   5046289,  -5963938,
              -5963948, -28836048]))

In [33]:

for i in range(length):
    x=int(ret[i])
    xi=(x/(1<<16))/256.0
    xr=(x%(1<<16))/256.0
    print(f"out[{i}]={xr:.4f}, {xi:.4f}")

out[0]=0.9102, 0.1211
out[1]=3.1523, -11.1757
out[2]=12.5000, -0.5467
out[3]=1.3125, -5.0703
out[4]=252.9531, -2.6368
out[5]=10.3086, -4.0662
out[6]=254.9102, -7.6914
out[7]=7.2266, 12.3204
out[8]=6.5859, -2.6093
out[9]=3.1562, 2.8086
out[10]=0.4336, -0.6797
out[11]=12.5898, 1.9260
out[12]=0.1055, -4.2266
out[13]=5.8477, 1.2852
out[14]=10.1836, 2.7814
out[15]=251.8398, -0.7540
out[16]=5.2617, -7.4570
out[17]=4.2305, 1.2227
out[18]=254.3906, 0.3789
out[19]=252.0312, -0.0821
out[20]=255.9844, 5.1211
out[21]=1.5430, -3.1758
out[22]=248.2148, -12.5822
out[23]=0.7109, 15.8438
out[24]=251.0078, -6.6759
out[25]=252.7266, 5.4765
out[26]=249.2383, 10.1210
out[27]=4.4102, 6.0274
out[28]=1.2227, 2.4063
out[29]=252.2852, 1.8203
out[30]=2.0977, -1.1093
out[31]=3.0039, 13.6719


In [38]:
f2 = open("dataOUT.txt", 'r')
out_vals=[x.split() for x in f2.readlines()]
error_count=0
for i in range(length):
    x=int(ret[i])
    xi=(x/(1<<16))/256.0
    xr=(x%(1<<16))/256.0
    if ((float(out_vals[i][0]) -xr) > 0.1 or (float(out_vals[i][1]) -xi)>0.1):
        print(float(out_vals[i][0]) , xr)
        error_count=error_count+1 
print("Output with "+str(error_count)+" errors")


Output with 0 errors
